# Web Scraping Demo: Menu Extraction with OCR

This notebook demonstrates how to use the `full_scrape_website` function to extract menu items from restaurant websites that display menus as images.

The implementation uses PaddleOCR to process menu images and extract dish names and prices.

In [ ]:
# Import necessary libraries
import pandas as pd
import sys
import os

# Add src directory to Python path
sys.path.append(os.path.join('..', 'src'))

from web_scraper import full_scrape_website

print("Web scraper imported successfully!")

## Usage Example

The main function `full_scrape_website` is designed to:

1. **Fetch webpage content** from the restaurant website
2. **Find menu images** - only processes images with 'menu', 'kaart', or 'speisen' in their URL (ignores logo images)
3. **Download and process images** using PaddleOCR
4. **Extract text** and group OCR results by Y-position to reconstruct lines
5. **Parse menu items** using regex to identify dish names and prices
6. **Return a DataFrame** with extracted menu items

### Key Features:
- **Robust image filtering**: Only processes relevant menu images
- **Debug output**: Prints all OCR text for troubleshooting
- **Flexible parsing**: Handles various price formats (€12.50, 12,50€, etc.)
- **Line reconstruction**: Groups OCR boxes by Y-position to rebuild text lines
- **Error handling**: Gracefully handles network issues and OCR failures

In [ ]:
# Example usage as specified in the problem statement
web_in = 'https://www.tvijverhof.be'

# Run the scraper
print(f"Scraping menu from: {web_in}")
df_test = full_scrape_website(web_in)

# Display results
print(f"\nFound {len(df_test)} menu items")
display(df_test)

## Expected Output Format

The function returns a pandas DataFrame with the following columns:
- `dish_name`: Name of the dish extracted from the menu
- `price`: Price of the dish (float) or None if not found

### Example Output:

| dish_name | price |
|-----------|-------|
| Schnitzel Wienerschnitzel | 18.50 |
| Pasta Carbonara | 14.90 |
| Caesar Salad | 12.00 |
| Beef Steak | 24.50 |


In [ ]:
# Display additional information about the results
if not df_test.empty:
    print("DataFrame Info:")
    print(df_test.info())
    
    print("\nSample menu items:")
    for idx, row in df_test.head(10).iterrows():
        dish_name = row['dish_name']
        price = row['price']
        if price is not None:
            print(f"• {dish_name} - €{price:.2f}")
        else:
            print(f"• {dish_name} - Price not found")
            
    # Basic statistics
    prices = df_test['price'].dropna()
    if len(prices) > 0:
        print(f"\nPrice Statistics:")
        print(f"Average price: €{prices.mean():.2f}")
        print(f"Min price: €{prices.min():.2f}")
        print(f"Max price: €{prices.max():.2f}")
        print(f"Items with prices: {len(prices)}/{len(df_test)}")
else:
    print("No menu items found. Possible reasons:")
    print("- Website structure doesn't match expected patterns")
    print("- No menu images found with expected keywords")
    print("- OCR unable to extract readable text")
    print("- Network connectivity issues")

## Technical Implementation Details

### Image Filtering
The scraper only processes images with specific keywords in their URLs:
- 'menu' (English)
- 'kaart' (Dutch)
- 'speisen' (German)

This ensures that only relevant menu images are processed, ignoring logos and other decorative images.

### OCR Processing
1. **Text Extraction**: Uses PaddleOCR to extract text with bounding boxes
2. **Confidence Filtering**: Only includes text with confidence > 0.5
3. **Line Grouping**: Groups text elements by Y-position (within 20 pixels)
4. **Line Reconstruction**: Sorts text within each line by X-position

### Price Detection
The scraper uses multiple regex patterns to detect prices:
- `€12.50` or `€ 12,50`
- `12.50€` or `12,50 €`
- `12.50 EUR`
- Numbers at the end of lines

### Error Handling
- Graceful degradation when PaddleOCR is not available
- Network timeout handling
- Temporary file cleanup
- Comprehensive logging for debugging

In [ ]:
# Test with debug output (if OCR is available)
print("\nDetailed Debug Information:")
print("The scraper provides detailed debug output including:")
print("- All OCR text extracted from images")
print("- Confidence scores for each text element")
print("- Line grouping results")
print("- Menu item parsing steps")
print("\nThis helps troubleshoot issues with menu extraction.")